## WHAT IS DONE IN PROJECT

1. Data cleaning
2. Exploratory data preparation
3. Airport choice model
4. Airline choice model
5. Logistic regression
6. Decision tree
7. Random forest
8. Model comparison
9. Feature importance analysis

# Airport and Airline Choice Modeling using Machine Learning

This project analyzes passenger decision behavior in airport and airline selection using survey data.

Two prediction tasks are performed:

1. Airport Choice Prediction
2. Airline Choice Prediction

Machine learning models such as Logistic Regression, Decision Tree, and Random Forest are used to identify the key factors influencing these choices.

In [8]:
import pandas as pd

df = pd.read_excel("airport_choice_survey_EN_ver2.0_Capstone.xlsx")

print("Shape:", df.shape)
print("\nFirst 5 columns:", df.columns.tolist()[:5])

print("\nAirport value counts:\n", df["Airport"].value_counts(dropna=False))
print("\nAirline value counts:\n", df["Airline"].value_counts(dropna=False))

Shape: (488, 28)

First 5 columns: ['ID', 'Airport', 'Airline', 'Age', 'Gender']

Airport value counts:
 Airport
2    249
1    239
Name: count, dtype: int64

Airline value counts:
 Airline
1.0    153
4.0    137
2.0    107
3.0     81
NaN     10
Name: count, dtype: int64


In [9]:
import numpy as np

# Make a copy so we don't damage original data
df_clean = df.copy()

# Drop rows where Airport is missing
df_clean = df_clean.dropna(subset=["Airport"])

# Convert Airport to integer
df_clean["Airport"] = df_clean["Airport"].astype(int)

print("New shape:", df_clean.shape)

# Check column data types
print("\nColumn types:\n")
print(df_clean.dtypes)

# Check if any columns contain mixed types
print("\nObject columns:\n")
print(df_clean.select_dtypes(include=["object"]).columns.tolist())

New shape: (488, 28)

Column types:

ID                             int64
Airport                        int64
Airline                      float64
Age                          float64
Gender                       float64
Nationality                    int64
TripPurpose                    int64
TripDuration                   int64
FlyingCompanion                int64
ProvinceResidence              int64
GroupTravel                    int64
NoTripsLastYear                int64
FrequentFlightDestination     object
Destination                  float64
FlightNo                      object
DepartureHr                   object
DepartureMn                  float64
DepartureTime                  int64
SeatClass                    float64
Airfare                      float64
NoTransport                    int64
ModeTransport                  int64
AccessCost                   float64
AccessTime                   float64
Occupation                     int64
Income                       float64
M

## Airport Choice Model

This model predicts which airport a passenger chooses based on travel characteristics, demographics, and cost factors.

Models tested:

• Logistic Regression  
• Decision Tree  
• Random Forest

In [11]:
# Define target
y = df_clean["Airport"]

# Define features
X = df_clean.drop(columns=[
    "Airport",   # target
    "Airline",   # second target
    "ID",        # identifier
    "FlightNo"   # not useful
])

print("X shape:", X.shape)
print("y shape:", y.shape)

# Identify numeric vs categorical

import numpy as np

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("\nNumeric columns:", num_cols)
print("\nCategorical columns:", cat_cols)

X shape: (488, 24)
y shape: (488,)

Numeric columns: ['Age', 'Gender', 'Nationality', 'TripPurpose', 'TripDuration', 'FlyingCompanion', 'ProvinceResidence', 'GroupTravel', 'NoTripsLastYear', 'Destination', 'DepartureMn', 'DepartureTime', 'SeatClass', 'Airfare', 'NoTransport', 'ModeTransport', 'AccessCost', 'AccessTime', 'Occupation', 'Income', 'Mileage']

Categorical columns: ['FrequentFlightDestination', 'DepartureHr', 'MileageAirline']


In [12]:
# Convert categorical columns to string
# (prevents int/str mixing error)
X[cat_cols] = X[cat_cols].astype("string")

# Fill missing categorical values
X[cat_cols] = X[cat_cols].fillna("Missing")

# Fill missing numeric values
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

print("Missing values after cleaning:")
print(X.isna().sum().sum())

Missing values after cleaning:
0


In [13]:
from sklearn.model_selection import train_test_split

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (366, 24)
Testing data: (122, 24)


In [14]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

print("Preprocessor created successfully")

Preprocessor created successfully


In [15]:
# Logistic 
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

logit_model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=3000))
])

# Train
logit_model.fit(X_train, y_train)

# Predict
pred = logit_model.predict(X_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
print("\nClassification Report:\n", classification_report(y_test, pred))

Confusion Matrix:
 [[49 11]
 [ 3 59]]

Classification Report:
               precision    recall  f1-score   support

           1       0.94      0.82      0.88        60
           2       0.84      0.95      0.89        62

    accuracy                           0.89       122
   macro avg       0.89      0.88      0.88       122
weighted avg       0.89      0.89      0.88       122



/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Interpretation:
49 passengers correctly predicted Airport 1

59 passengers correctly predicted Airport 2

Only 14 mistakes total

### Meaning:
Model predicts Airport 1 very precisely

Model detects Airport 2 very well

In [17]:
from sklearn.metrics import roc_auc_score

proba = logit_model.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, proba)

print("ROC-AUC:", auc)

ROC-AUC: 0.9225806451612903


### Interpretation:

AUC = 0.92

This means the model can distinguish Airport 1 vs Airport 2 about 92% of the time

So your airport choice model is performing very well.

In [19]:
# Top Influential Features
# Get feature names after encoding

feature_names = logit_model.named_steps["prep"].get_feature_names_out()

# Get coefficients from logistic regression
coefficients = logit_model.named_steps["clf"].coef_[0]

# Create dataframe
import pandas as pd

importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

# Sort by strongest influence
importance = importance.sort_values(by="Coefficient", key=abs, ascending=False)

importance.head(15)

,Feature,Coefficient
64,num__Destination,-1.065792
66,num__DepartureTime,-0.897180
0,cat__FrequentFlightDestination_1,-0.706823
54,cat__MileageAirline_Missing,-0.599669
12,cat__FrequentFlightDestination_3,0.490309
27,cat__DepartureHr_16,0.450198
58,num__TripPurpose,0.410951
57,num__Nationality,0.385811
69,num__NoTransport,-0.357667
51,cat__MileageAirline_4,0.322062


### Meaning:

Positive value → increases probability of Airport 2

Negative value → increases probability of Airport 1

In [21]:
# Decision Tree Model
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report

tree_model = Pipeline([
    ("prep", preprocessor),
    ("clf", DecisionTreeClassifier(max_depth=5, random_state=42))
])

tree_model.fit(X_train, y_train)

pred_tree = tree_model.predict(X_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, pred_tree))
print("\nClassification Report:\n", classification_report(y_test, pred_tree))

Confusion Matrix:
 [[50 10]
 [ 5 57]]

Classification Report:
               precision    recall  f1-score   support

           1       0.91      0.83      0.87        60
           2       0.85      0.92      0.88        62

    accuracy                           0.88       122
   macro avg       0.88      0.88      0.88       122
weighted avg       0.88      0.88      0.88       122



Logistic regression is slightly better, but both are strong. As Accuracy for Logistic is 0.89 and Decision Tree is 0.88.

In [23]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=300, random_state=42))
])

rf_model.fit(X_train, y_train)

pred_rf = rf_model.predict(X_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, pred_rf))
print("\nClassification Report:\n", classification_report(y_test, pred_rf))

Confusion Matrix:
 [[52  8]
 [ 1 61]]

Classification Report:
               precision    recall  f1-score   support

           1       0.98      0.87      0.92        60
           2       0.88      0.98      0.93        62

    accuracy                           0.93       122
   macro avg       0.93      0.93      0.93       122
weighted avg       0.93      0.93      0.93       122



Random Forest achieved the highest predictive accuracy, while logistic regression provides better interpretability of the factors affecting airport choice.

In [25]:
# Random Forest Feature Importance
# Extract feature names
feature_names = rf_model.named_steps["prep"].get_feature_names_out()

# Extract importance
importances = rf_model.named_steps["clf"].feature_importances_

import pandas as pd

rf_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

rf_importance = rf_importance.sort_values(by="Importance", ascending=False)

rf_importance.head(15)

,Feature,Importance
64,num__Destination,0.128611
59,num__TripDuration,0.068696
65,num__DepartureMn,0.061965
0,cat__FrequentFlightDestination_1,0.056537
73,num__Occupation,0.052602
66,num__DepartureTime,0.048265
74,num__Income,0.038365
55,num__Age,0.036272
54,cat__MileageAirline_Missing,0.034934
71,num__AccessCost,0.033994


# Airline Choice Model

This section analyzes passengers' airline selection behavior.

Target variable: **Airline**

Airline categories:

1 = Korean Air  
2 = Asiana  
3 = Korean Low-Cost Carrier  
4 = Foreign Airline  

Models used:
- Multinomial Logistic Regression
- Random Forest


In [27]:
# Prepare Airline Model
# Prepare dataset for Airline model

df_air = df_clean.dropna(subset=["Airline"]).copy()

y_air = df_air["Airline"]

X_air = df_air.drop(columns=[
    "Airline",
    "Airport",
    "ID",
    "FlightNo"
])

print("X_air shape:", X_air.shape)
print("y_air shape:", y_air.shape)

print("\nAirline distribution:")
print(y_air.value_counts())

X_air shape: (478, 24)
y_air shape: (478,)

Airline distribution:
Airline
1.0    153
4.0    137
2.0    107
3.0     81
Name: count, dtype: int64


In [28]:
# Prepare Features for Airline Model
import numpy as np

# Identify numeric and categorical columns again
num_cols_air = X_air.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_air = X_air.select_dtypes(include=["object","string"]).columns.tolist()

print("Numeric columns:", num_cols_air)
print("\nCategorical columns:", cat_cols_air)

# Convert categorical columns to string
X_air[cat_cols_air] = X_air[cat_cols_air].astype("string")

# Fill missing categorical
X_air[cat_cols_air] = X_air[cat_cols_air].fillna("Missing")

# Fill numeric missing
for col in num_cols_air:
    X_air[col] = X_air[col].fillna(X_air[col].median())

print("\nMissing values remaining:", X_air.isna().sum().sum())

Numeric columns: ['Age', 'Gender', 'Nationality', 'TripPurpose', 'TripDuration', 'FlyingCompanion', 'ProvinceResidence', 'GroupTravel', 'NoTripsLastYear', 'Destination', 'DepartureMn', 'DepartureTime', 'SeatClass', 'Airfare', 'NoTransport', 'ModeTransport', 'AccessCost', 'AccessTime', 'Occupation', 'Income', 'Mileage']

Categorical columns: ['FrequentFlightDestination', 'DepartureHr', 'MileageAirline']

Missing values remaining: 0


In [29]:
# Train/Test Split (Airline Model)
from sklearn.model_selection import train_test_split

X_train_air, X_test_air, y_train_air, y_test_air = train_test_split(
    X_air,
    y_air,
    test_size=0.25,
    random_state=42,
    stratify=y_air
)

print("Training data:", X_train_air.shape)
print("Testing data:", X_test_air.shape)

Training data: (358, 24)
Testing data: (120, 24)


In [30]:
# Preprocessing Pipeline (Airline Model)
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor_air = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_air),
        ("num", "passthrough", num_cols_air)
    ]
)

print("Preprocessor ready")

Preprocessor ready


In [31]:
# Multinomial Logistic Regression (Airline Choice)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

logit_air = Pipeline([
    ("prep", preprocessor_air),
    ("clf", LogisticRegression(max_iter=4000))
])

logit_air.fit(X_train_air, y_train_air)

pred_air = logit_air.predict(X_test_air)

print("Confusion Matrix:\n", confusion_matrix(y_test_air, pred_air))
print("\nClassification Report:\n", classification_report(y_test_air, pred_air))

Confusion Matrix:
 [[19  5  2 13]
 [ 3  9  2 13]
 [ 7  4  1  8]
 [11  3  0 20]]

Classification Report:
               precision    recall  f1-score   support

         1.0       0.47      0.49      0.48        39
         2.0       0.43      0.33      0.38        27
         3.0       0.20      0.05      0.08        20
         4.0       0.37      0.59      0.45        34

    accuracy                           0.41       120
   macro avg       0.37      0.36      0.35       120
weighted avg       0.39      0.41      0.38       120



/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Confusion Matrix Interpretation
Airline 1 - 19 / 39

Airline 2 -  9 / 27

Airline 3 -  1 / 20

Airline 4 -  20 / 34

Airline 3 (LCC) is hardest to predict — which is common in airline datasets.

In [33]:
# Random Forest Airline Model

from sklearn.ensemble import RandomForestClassifier

rf_air = Pipeline([
    ("prep", preprocessor_air),
    ("clf", RandomForestClassifier(n_estimators=400, random_state=42))
])

rf_air.fit(X_train_air, y_train_air)

pred_rf_air = rf_air.predict(X_test_air)

print("Confusion Matrix:\n", confusion_matrix(y_test_air, pred_rf_air))
print("\nClassification Report:\n", classification_report(y_test_air, pred_rf_air))

Confusion Matrix:
 [[28  1  3  7]
 [ 5 13  1  8]
 [ 0  2 17  1]
 [ 7  4  1 22]]

Classification Report:
               precision    recall  f1-score   support

         1.0       0.70      0.72      0.71        39
         2.0       0.65      0.48      0.55        27
         3.0       0.77      0.85      0.81        20
         4.0       0.58      0.65      0.61        34

    accuracy                           0.67       120
   macro avg       0.68      0.67      0.67       120
weighted avg       0.67      0.67      0.66       120



Random Forest improved prediction by 26%.

Best Predicted Airline - 

Airline 3 (LCC)

Recall = 0.85

# Feature Importance Analysis

Random Forest feature importance is used to identify the most influential factors affecting passenger airport and airline choice.

In [36]:
# Extract feature names
feature_names_air = rf_air.named_steps["prep"].get_feature_names_out()

# Feature importance
importances_air = rf_air.named_steps["clf"].feature_importances_

import pandas as pd

importance_air = pd.DataFrame({
    "Feature": feature_names_air,
    "Importance": importances_air
})

importance_air = importance_air.sort_values(by="Importance", ascending=False)

importance_air.head(15)

,Feature,Importance
52,num__Age,0.058691
61,num__Destination,0.057149
65,num__Airfare,0.056571
62,num__DepartureMn,0.056318
56,num__TripDuration,0.051449
68,num__AccessCost,0.046852
69,num__AccessTime,0.045622
57,num__FlyingCompanion,0.045617
60,num__NoTripsLastYear,0.038084
67,num__ModeTransport,0.037783


# Interpretation

Passengers choose airlines mainly based on:

### Economic factors
Airfare -
Access cost -
Income

### Travel characteristics

Destination - 
Trip duration - Departure time

### Passenger characteristics

Age - 
Travel frequency - 
Travel companions

These are very common findings in airline choice research, so your results are realistic.

# Best Model For Airport Choice- Random Forest 93% Accuracy

## Key factors:

Destination - Departure time - Trip duration - Access cost - Access time

# Best Model For Airline Choice- Random Forest 67% Accuracy
## Key factors:

Age - Destination - Airfare - Trip duration - Access cost - Travel frequency

## Conclusion

The analysis shows that travel-related factors such as destination, trip duration, airfare, and departure time play an important role in airport and airline selection.

Random Forest models performed best for both prediction tasks, indicating that nonlinear relationships exist between passenger characteristics and travel decisions.

These results can help airports and airlines better understand passenger preferences and improve service planning.